In [0]:
df_movies = spark.read.csv("/Volumes/workspace/default/moviesdata/movies.csv", header=True, inferSchema=True)
df_ratings = spark.read.csv("/Volumes/workspace/default/moviesdata/ratings.csv", header=True, inferSchema=True)
df_tags = spark.read.csv("/Volumes/workspace/default/moviesdata/tags.csv", header=True, inferSchema=True)
df_links = spark.read.csv("/Volumes/workspace/default/moviesdata/links.csv", header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import when, col

df_movies_null = df_movies.withColumn(
    "title",
    when(col("movieId") % 10 == 0, None).otherwise(col("title"))
).withColumn(
    "genres",
    when(col("movieId") % 15 == 0, None).otherwise(col("genres"))
)

In [0]:
df_movies_null.display()

In [0]:
df_ratings_null = df_ratings.withColumn(
    "rating",
    when(col("rating") % 3 == 0, None).otherwise(col("rating"))
)

In [0]:
df_ratings_null.display()

In [0]:
df_ratings_dup = df_ratings_null.union(df_ratings_null.limit(100))

In [0]:
df_ratings_dup.display()

**checking duplicate rows**

In [0]:
df_ratings_dup.groupBy("userId", "movieId", "rating", "timestamp") \
    .count() \
    .filter("count > 1") \
    .show()

In [0]:
df_movies_dup = df_movies_null.union(df_movies_null.limit(50))

In [0]:
from pyspark.sql.functions import lit

df_ratings_dirty = df_ratings_dup.withColumn(
    "rating",
    when(col("userId") % 30 == 0, 7.5)  # invalid rating (>5)
    .otherwise(col("rating"))
)

**wrong format**

In [0]:
from pyspark.sql.functions import col, when

df_links_dirty = df_links.withColumn(
    "imdbId",
    when(col("movieId").cast("int") % 20 == 0, "unknown")
    .otherwise(col("imdbId"))
)

In [0]:
df_links_dirty.display()

In [0]:
# Paths
raw_path = "/Volumes/workspace/default/moviesdata"
bronze_path = "/Volumes/workspace/default/moviesdirtydata"

# 1. READ RAW CSV FILES
df_movies = spark.read.option("header", True).csv(f"{raw_path}/movies.csv")
df_ratings = spark.read.option("header", True).csv(f"{raw_path}/ratings.csv")
df_tags = spark.read.option("header", True).csv(f"{raw_path}/tags.csv")
df_links = spark.read.option("header", True).csv(f"{raw_path}/links.csv")

# 3. SAVE AS DELTA (IN SEPARATE FOLDERS)
df_movies_null.write.mode("overwrite").format("delta") \
    .save(f"{bronze_path}/movies")

df_ratings_dirty.write.mode("overwrite").format("delta") \
    .save(f"{bronze_path}/ratings")

df_tags.write.mode("overwrite").format("delta") \
    .save(f"{bronze_path}/tags")

df_links_dirty.write.mode("overwrite").format("delta") \
    .save(f"{bronze_path}/links")


In [0]:
df_ratings_dirty.write.mode("overwrite").format("delta") \
    .save("/Volumes/workspace/default/moviesdirtydata/ratings")

In [0]:
df_links_dirty.write.mode("overwrite").format("delta") \
    .save(f"{bronze_path}/links")

In [0]:
df_check = spark.read.format("delta") \
    .load("/Volumes/workspace/default/moviesdirtydata/links")

df_check.filter(df_check.imdbId == "unknown").show()

In [0]:
df_links_dirty = df_links.withColumn(
    "imdbId",
    when(col("movieId").cast("int") % 20 == 0, "unknown")
    .otherwise(col("imdbId"))
)

df_links_dirty.show()

In [0]:
df_links_dirty.write.mode("overwrite").format("delta") \
    .save("/Volumes/workspace/default/moviesdirtydata/links")